In [ ]:
%matplotlib inline
import numpy as np
import os
import itertools
import torch
import h5py
import matplotlib.pyplot as plt
import sys


sys.path.append('../../../')
from libs import cond_gen
from libs.shap_eval import plot_shap_distributions, plot_shap_std_deviations, plot_shap_means, print_top_features, generate_all_shap_mats, generate_sensors_per_batch, create_shap_masks, normalize_shap_values

In [ ]:
dir_path = "../"  # relative to this notebook: osp_utils/shap/
filename = "shap-M30.0%-from_ground_and_wall-t_sub-1000-T_sub_mapgd-20-gditer-50-start{start_value}-stop{stop_value}-step50-ncoali{coalitions}.npz"

results_path = dir_path + 'results/shaply-values/v5.5-tdiff20-gd50-snaps-25000-step50/'
npz_files = [os.path.join(results_path, file) for file in os.listdir(results_path) if file.endswith('.npz')]

In [ ]:
len(npz_files)

In [ ]:
def load_shap_values(start=None, stop=None, coalitions=None, step=1, bs=1, results_path=None, filename=None):
    """
    Load SHAP values from multiple files in a specified range and concatenate them into a single array.
    
    Parameters:
    -----------
    start : int
        The starting value for the range of files to process.
    stop : int
        The stopping value for the range of files to process (exclusive).
    step : int, optional
        The step size for the range of files to process (default is 1).
    bs : int, optional
        The batch size for file grouping (default is 1).
    results_path : str
        The path to the directory containing the SHAP value files.
    filename : str
        The filename format, which must include placeholders for `start_value` and `stop_value`.
        Example: "shap_{start_value}_{stop_value}.npz"
    
    Returns:
    --------
    all_shaps : np.ndarray
        A concatenated array of SHAP values across all processed files.
    seg_tensor : np.ndarray
        The segmentation tensor from the last successfully loaded file.
    x : np.ndarray
        The x-coordinates from the last successfully loaded file.
    y : np.ndarray
        The y-coordinates from the last successfully loaded file.
    
    Notes:
    ------
    - If a file does not exist or an error occurs during loading, it is skipped, and the error is logged.
    - The function returns data from the last successfully loaded file for `seg_tensor`, `x`, and `y`.
    """
    if results_path is None or filename is None:
        raise ValueError("Both `results_path` and `filename` must be specified.")
    
    counter = 0
    all_shaps = []
    seg_tensor, x, y = None, None, None  # Initialize to ensure they exist
    
    for i in range(start, stop, bs * step):
        stop_value = i + bs * step
        file_path = os.path.join(results_path, filename.format(start_value=i, stop_value=stop_value, coalitions=coalitions))
        
        if os.path.exists(file_path):
            try:
                loaded_data = np.load(file_path)
                shaps = loaded_data['shaps']
                
                # Load variables (but only keep necessary ones)
                seg_tensor = loaded_data.get('seg_tensor', None)
                x = loaded_data.get('x', None)
                y = loaded_data.get('y', None)
                
                counter += 1
                all_shaps.append(shaps.reshape(-1, 1))
            except Exception as e:
                print(f"Error loading file {file_path}: {e}")
        else:
            print(f"File {file_path} does not exist.")
    
    if not all_shaps:
        raise RuntimeError("No SHAP values were loaded. Please check the input parameters and files.")
    
    all_shaps = np.abs(np.concatenate(all_shaps, axis=1))
    return all_shaps, seg_tensor, x, y

In [ ]:
all_shaps, seg_tensor, x, y = load_shap_values(start=0, stop=25000, coalitions=3000, step=50, bs=1, results_path=results_path ,filename=filename)

In [ ]:
unique_values = np.unique(seg_tensor, return_counts=False)
unique_values = unique_values[np.where(unique_values != 0.0)]
nmax2= len(unique_values) 

In [ ]:
shap_data_dict = {
    "coalitions" : all_shaps[:,:1],
}


figname = "compare-v5.5"

In [ ]:
plot_shap_distributions(shap_data_dict=shap_data_dict, alpha=0.3, ylim=(0, 4), feature_wise_norm=True, figname=figname, out_show=True, out_pdf=False)

In [ ]:
plot_shap_means(shap_data_dict=shap_data_dict, feature_wise_norm=True, figname=figname, out_show=True, out_pdf=False)

In [ ]:
print_top_features(shap_data_dict=shap_data_dict, k=10, feature_wise_norm=True)

In [ ]:
all_shap_mats = generate_all_shap_mats( all_shaps=all_shaps,
                                            seg_tensor=seg_tensor,
                                            mask=30,
                                            x_axis=x,
                                            y_axis=y,
                                            feature_wise_norm=True,
                                            plot_shap_mats=False
)



In [ ]:
threshold = 0.73

strategy="mean"  #"mean" #"mean_plus_std" #"MultiMode"

final_sensor_array, reduced_shap_mask = create_shap_masks(all_shap_mats=all_shap_mats, 
                strategy=strategy, threshold=threshold, mask_value=30,
                 x_axis=x, y_axis=y, plot_final_mask=True)



In [ ]:
np.savez(f'shap_v5.5_mask30-thresh{threshold}-strategy-{strategy}.npz',shap_mask = final_sensor_array, reduced_shap_mask = reduced_shap_mask ,mask=30, x_axis=x, y_axis=y, mask_type=f"osp-t{threshold}-v5.5", threshold=threshold ) 

In [ ]:
np.savez(f'shap_v5.5_shap_values.npz',shap_mats =all_shap_mats ,mask=30, x_axis=x, y_axis=y) 
